# Client node to handle the robot goals

In the next 3 cells there is the code to create a client for the action server that handles the movements of the robot.

This client gives to the user the possibility to chose new goals and cancel the current goal.

It also shows some information about the robot.

The first cell imports some libraries used by the code and creates some global variables.

The second cell creates the functions used by the code.

The third cell runs the main code of the node.

All the cells in this notebook should be runned in order.

In [1]:
#!/usr/bin/env python3

import rospy
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PlanningFeedback
from nav_msgs.msg import Odometry
from assignment_2_part1.msg import RobotState  # Custom message
from geometry_msgs.msg import Point
import threading
import ipywidgets as widgets

# === Global Variables ===
cnt = 0
current_goal_x = 0
current_goal_y = 0

# For plotting number of reached / not-reached targets
reached = 0
not_reached = 0
goal_state = 1 # starts at one to avoid counting a false not_reached goal at start

# These will be initialized in main()
ac = None
state_pub = None
coord_pub = None

# Create an output widget
output = widgets.Output(layout=widgets.Layout(display='flex', justify_content='center'))

In [2]:
def feedback_callback(feedback: PlanningFeedback):
    global current_goal_x, current_goal_y, cnt, reached, goal_state
    threshold = 0.5
    if abs(feedback.actual_pose.position.x - current_goal_x) < threshold and \
       abs(feedback.actual_pose.position.y - current_goal_y) < threshold and cnt < 1:
        rospy.loginfo("Target successfully reached!")
        cnt += 1
        reached += 1
        goal_state = 1
        plot_reach_status()

def odom_callback(msg: Odometry):
    state_msg = RobotState()
    state_msg.x = msg.pose.pose.position.x
    state_msg.y = msg.pose.pose.position.y
    state_msg.vel_x = msg.twist.twist.linear.x
    state_msg.vel_z = msg.twist.twist.angular.z
    state_pub.publish(state_msg)
    
# Plotting function
def plot_reach_status():
    with output:
        clear_output(wait=True)
        labels = ['Reached', 'Not Reached']
        values = [reached, not_reached]
        colors = ['green', 'red']
        
        plt.figure(figsize=(5, 4))
        plt.title("Target Reach Status")
        plt.bar(labels, values, color=colors)
        plt.ylim(0, max(1, reached + not_reached))  # avoid 0-height y-axis
        plt.show()
    
def spin_thread():
    rospy.spin()

In [3]:
#global ac, state_pub, current_goal_x, current_goal_y, cnt, coord_pub, vis
    
rospy.init_node("client_to_robotac")
    
state_pub = rospy.Publisher("/robot_state", RobotState, queue_size=10)
coord_pub = rospy.Publisher("/targ_coords", Point, queue_size=10)
rospy.Subscriber("/odom", Odometry, odom_callback)
    
ac = actionlib.SimpleActionClient("/reaching_goal", PlanningAction)
rospy.loginfo("Waiting for action server to start.")
ac.wait_for_server()
rospy.loginfo("Action server started, waiting for user input.")
    
thread = threading.Thread(target=spin_thread)
thread.daemon = True
thread.start()

[INFO] [1746978974.612450, 4854.417000]: Waiting for action server to start.
[INFO] [1746978974.652227, 4854.464000]: Action server started, waiting for user input.


# Interface to send a new goal and to cancel the current goal

In the following cell a user interface to send and cancel goal is created.

This inteface has 2 sliders each one connected to his own textbox to chose the value of the goal that will be sent to the robot.

After choosing the desired value the user can press the "Submit new goal button" to send the new goal.

This interface also provides a button to cancel the current goal and stop the robot in its current position.


In [4]:
import ipywidgets as widgets
from IPython.display import display

# Function to be called on button click
def on_button_click_send_new_goal(b):
    global ac, current_goal_x, current_goal_y, cnt, coord_pub, goal_state, not_reached
    
    
    if goal_state == 0: # we are sending a new target and the previous one has not been reached, count it as not reached
        not_reached += 1
        plot_reach_status()
    
    goal_state = 0 # reset since the new state has not been reached yet
    
    cnt = 0
    goal = PlanningGoal()
    
    goal.target_pose.pose.position.x = x.value
    goal.target_pose.pose.position.y = y.value
    
    current_goal_x = goal.target_pose.pose.position.x
    current_goal_y = goal.target_pose.pose.position.y
            
    coord_msg = Point()
    coord_msg.x = current_goal_x
    coord_msg.y = current_goal_y
    coord_pub.publish(coord_msg)
            
    ac.send_goal(goal, feedback_cb=feedback_callback)
    rospy.loginfo("Sending new goal.")
    
# Function to be called on button click
def on_button_click_cancel_goal(b):
    global ac, not_reached, goal_state
    rospy.loginfo("Canceling the current goal.")
    ac.cancel_goal()
    if goal_state == 0: # we have to count a goal as not reached only if it has not been reached, not every time that someone presses the button
        not_reached += 1 # when we cancel the current goal we count it as not reached
        goal_state = 1 # to avoid a double press of the btn and eventual wrong increase of not_reached goals
        plot_reach_status()

# First pair of widgets (X)
x = widgets.FloatText()
x_slider = widgets.FloatSlider(
    continuous_update=False,
    min=-10,
    max=10
)
widgets.jslink((x, 'value'), (x_slider, 'value'))

box1 = widgets.VBox([
    widgets.HTML("<h4>Set X coordinate:</h4>"),
    x,
    x_slider
])

# Second pair of widgets (Y)
y = widgets.FloatText()
y_slider = widgets.FloatSlider(
    continuous_update=False,
    min=-10,
    max=10
)
widgets.jslink((y, 'value'), (y_slider, 'value'))

box2 = widgets.VBox([
    widgets.HTML("<h4>Set Y coordinate:</h4>"),
    y,
    y_slider
])

# Button with spacing
set_goal_button = widgets.Button(
    description="Submit new Goal",
    button_style='success',
    layout=widgets.Layout(margin='30px 0 0 0', width='300px', height='50px')  # Top margin
)

# Button with spacing
cancel_goal_button = widgets.Button(
    description="Cancel actual Goal",
    button_style='danger',
    layout=widgets.Layout(margin='30px 0 0 0', width='300px', height='50px')  # Top margin
)

# Center the button using HBox
centered_button = widgets.HBox([set_goal_button], layout=widgets.Layout(justify_content='center'))

set_goal_button.on_click(on_button_click_send_new_goal)

# Center the button using HBox
centered_button_cancel = widgets.HBox([cancel_goal_button], layout=widgets.Layout(justify_content='center'))

cancel_goal_button.on_click(on_button_click_cancel_goal)

# Display everything together
display(widgets.VBox([
    widgets.Box([box1, box2], layout=widgets.Layout(justify_content='space-between', gap='50px')),
    centered_button, centered_button_cancel
]))

# Info about closest obstacle

In the following cell there is the code to subscribe to the topic of the laser.

The data received is then printed in real time to show the distance from the closest detected obstacle

In [5]:
from sensor_msgs.msg import LaserScan

label = widgets.HTML(
    value="<b style='font-size:24px;'>Closest Obstacle (m):</b>",
    layout=widgets.Layout(margin='20px 10px 0 20px', width='250px')
)

# FloatText input
closest_distance = widgets.FloatText(
    value=0.0,
    disabled=True,
    layout=widgets.Layout(width='80px', margin='20px 0 0 0')
)

def scan_callback(msg: LaserScan):
    if msg.ranges:
        min_dist = min([r for r in msg.ranges if r > 0])  # Avoid invalid 0 or inf
        closest_distance.value = round(min_dist, 2)

rospy.Subscriber("/scan", LaserScan, scan_callback)

# Horizontal box to align label and input
input_box = widgets.HBox([label, closest_distance])
display(input_box)

# Histogram of the number of reached and not reached goals

The following cell creates an histogram that counts the number of non reached and reached goals.

The histogram is updated real-time, when a goal is cancelled or overwrited the non reached goals bar will increment by one, the same thing happens to the reached goals bar when a goal is reached.

In [6]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# Display the interface
display(widgets.VBox([output], layout=widgets.Layout(align_items='center')))

plot_reach_status()

# Real time plot of robot position

In the following cell there is the code to generate a real time plot of the position of the robot.

In [7]:
%matplotlib widget

import matplotlib.pyplot as plt 
import rospy
import tf
from nav_msgs.msg import Odometry
from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation

class Visualiser:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ln, = plt.plot([], [], 'ro')
        self.x_data, self.y_data = [] , []

    def plot_init(self):
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        return self.ln,

    def odom_callback(self, msg):
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,


# rospy.init_node('odom_visualizer_node') 
vis = Visualiser()
sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)

ani = FuncAnimation(vis.fig, vis.update_plot, init_func=vis.plot_init) 
plt.show(block=True)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …